In [1]:
    def finalize_template_events(self):
        import numpy
        # Check that none of the template events have the same time index as an
        # existing event in events. I.e. don't list the same ifo event multiple
        # times when looping over sky points and time slides.
        existing_times = {}
        new_times = {}
        existing_template_id = {}
        new_template_id = {}
        existing_events_mask = {}
        new_template_event_mask = {}
        existing_template_event_mask = {}
        for i, ifo in enumerate(self.ifos):
            ifo_events = numpy.where(self.events['ifo'] == i)
            existing_times[ifo] = self.events['time_index'][ifo_events]
            new_times[ifo] = self.template_event_dict[ifo]['time_index']
            existing_template_id[ifo] = self.events['template_id'][ifo_events]
            new_template_id[ifo] = self.template_event_dict[ifo]['template_id']
            # This is true for each existing event that has the same time index
            # and template id as a template trigger.
            existing_events_mask[ifo] = numpy.argwhere(
                numpy.logical_and(
                    numpy.isin(existing_times[ifo], new_times[ifo]),
                    numpy.isin(existing_template_id[ifo], new_template_id[ifo])
                                 )).reshape(-1,)
            # This is true for each template event that has either a new
            # trigger time or a new template id.
            new_template_event_mask[ifo] = numpy.argwhere(
                numpy.logical_or(
                   ~numpy.isin(new_times[ifo], existing_times[ifo]),
                   ~numpy.isin(new_template_id[ifo], existing_template_id[ifo])
                                )).reshape(-1,)
            # This is true for each template event that has the same time index
            # and template id as an exisitng event trigger.
            existing_template_event_mask[ifo] = numpy.argwhere(
                numpy.logical_and(
                    numpy.isin(new_times[ifo], existing_times[ifo]),
                    numpy.isin(new_template_id[ifo], existing_template_id[ifo])
                                 )).reshape(-1,)
            # Set ids (These show how each trigger in the single ifo trigger
            # list correspond to the network triggers)
            num_events = len(new_template_event_mask[ifo])
            new_event_ids = numpy.arange(self.event_index[ifo],
                                         self.event_index[ifo] + num_events)
            # Every template event that corresponds to a new trigger gets a new
            # id. Triggers that have been found before are not saved.
#             import pdb
#             pdb.set_trace()
            self.template_event_dict[ifo]['event_id'][
                new_template_event_mask[ifo]] = new_event_ids
            self.template_event_dict['network'][ifo + '_event_id'][
                new_template_event_mask[ifo]] = new_event_ids
            # Template events that have been found before get the event id of
            # the first time they were found.
            self.template_event_dict['network'][ifo + '_event_id'][
                  existing_template_event_mask[ifo]] = \
                self.events[self.events['ifo'] == i][
                  existing_events_mask[ifo]]['event_id']
            self.event_index[ifo] = self.event_index[ifo] + num_events

        # Add the network event ids for the events with this template.
        num_events = len(self.template_event_dict['network'])
        new_event_ids = numpy.arange(self.event_index['network'],
                                     self.event_index['network'] + num_events)
        self.event_index['network'] = self.event_index['network'] + num_events
        self.template_event_dict['network']['event_id'] = new_event_ids
        # Move template events for each ifo to the events list
        for ifo in self.ifos:
            self.events = numpy.append(
                self.events,
                self.template_event_dict[ifo][new_template_event_mask[ifo]]
            )
            self.template_event_dict[ifo] = \
                                        numpy.array([], dtype=self.event_dtype)
        # Move the template events for the network to the network events list
        self.network_events = numpy.append(self.network_events,
                                   self.template_event_dict['network'])
        self.template_event_dict['network'] = \
                                numpy.array([], dtype=self.network_event_dtype)

In [2]:
import numpy as np
from pycbc.types import zeros, float32, complex64
from pycbc.events import EventManagerHM
from collections import defaultdict

arg_dict = {'verbose': True, 'output': '/Users/camill/projects/pycbc/test_hm/test.hdf', 'instruments': ['H1', 'L1', 'V1'], 'bank_file': '/Users/camill/projects/pycbc/test_hm/bank/mini-bank.hdf', 'snr_threshold': 4.0, 'low_frequency_cutoff': 20.0, 'approximant': ['IMRPhenomXHM:mtotal<4', 'IMRPhenomXHM:else'], 'order': '-1', 'taper_template': None, 'cluster_method': 'window', 'cluster_window': 0.0, 'bank_veto_bank_file': None, 'downsample_factor': 1, 'upsample_threshold': None, 'upsample_method': 'pruned_fft', 'user_tag': None, 'coinc_threshold': 7.0, 'timing_error': 0.005, 'num_timeslides': 1, 'channel_name':  {'H1': 'H1:GWOSC-4KHZ_R1_STRAIN', 'L1': 'L1:GWOSC-4KHZ_R1_STRAIN', 'V1': 'V1:GWOSC-4KHZ_R1_STRAIN'}, 'frame_files':  {'H1': ['./strain/H-H1_GWOSC_O3b_4KHZ_R1-1267732480-4096.gwf'], 'L1': ['./strain/L-L1_GWOSC_O3b_4KHZ_R1-1267732480-4096.gwf'], 'V1': ['./strain/V-V1_GWOSC_O3b_4KHZ_R1-1267732480-4096.gwf']}, 'psdvar_segment': None, 'psdvar_short_segment': None, 'psdvar_long_segment': None, 'psdvar_psd_duration': None, 'psdvar_psd_stride': None, 'psdvar_low_freq': None, 'psdvar_high_freq': None, 'processing_scheme': 'cpu', 'processing_device_id': 0, 'fft_backends': [], 'fftw_measure_level': 0, 'fftw_threads_backend': None, 'fftw_input_float_wisdom_file': None, 'fftw_input_double_wisdom_file': None, 'fftw_output_float_wisdom_file': None, 'fftw_output_double_wisdom_file': None, 'fftw_import_system_wisdom': False, 'cpu_affinity': None, 'cpu_affinity_from_env': None, 'trig_start_time':None, 'trig_end_time':None}
arg_dict["sample_rate"] = defaultdict(lambda: 512)
arg_dict["gps_start_time"] = defaultdict(lambda: 1234)
arg_dict["gps_end_time"] = defaultdict(lambda: 1235)
arg_dict["segment_start_pad"] = defaultdict(lambda: 0)
arg_dict["segment_end_pad"] = defaultdict(lambda: 0)

class Args(object):
    def __init__(self, arg_dict):
        for k,v in arg_dict.items():
            setattr(self, k, v)
    def __getitem__(self, item):
        return getattr(self, item)
args = Args(arg_dict)

ifo_list = args.instruments
# Put the ifos in alphabetical order so they are always called in
# the same order.
ifo_list.sort()
nifo = len(ifo_list)

ifo_out_types = {
    'time_index': int,
    'ifo': int, # IFO is stored as an int internally!
    'snr_dominant': complex64,
    'snr_subdominant': complex64,
    'sigma_sub_perp': float32,
    }
ifo_out_vals = {
    'time_index': None,
    'ifo': None,
    'snr_dominant': None,
    'snr_subdominant': None,
    'sigma_sub_perp': None,
    }
ifo_out_names = sorted(ifo_out_vals.keys())
network_out_types = {
    'snr_2_filter_rss': float32,
    'snr_2_filter': float32,
    'nifo': int,
    'timeslide_id': int,
    }
network_out_vals = {
    'snr_2_filter_rss': None,
    'snr_2_filter': None,
    'nifo': None,
    'timeslide_id': None,
    }
network_names = sorted(network_out_vals.keys())

In [3]:
event_mgr = EventManagerHM(
    args, ifo_list, ifo_out_names,
    [ifo_out_types[n] for n in ifo_out_names], network_names,
    [network_out_types[n] for n in network_names])

In [4]:
tmplt={"mass1":10, "mass2":10, "template_hash":123, "template_duration":1}
tmplt = Args(tmplt)
event_mgr.new_template(tmplt=tmplt,
    sigmasq_dom=defaultdict(lambda:100), sigmasq_sub=defaultdict(lambda:50))

In [5]:
coinc_idx = {
    "H1":np.array([0,1,2,3]),
    "L1":np.array([2,2,2,3]),
    "V1":np.array([1,0,1,3]),
}
snr_dom_dict = {
    "H1":6*np.array([0,1,2,3]),
    "L1":6*np.array([2,2,2,3]),
    "V1":6*np.array([1,0,1,3]),
}
snr_2_filter = np.array([10,12,7,8])

In [6]:
num_events = 4
for timeslide_id in [0,1]:
    for ifo in ifo_list:
        ifo_out_vals['time_index'] = (
            np.int64(coinc_idx[ifo])
            )
        ifo_out_vals['snr_dominant'] = abs(snr_dom_dict[ifo][coinc_idx[ifo]])
        ifo_out_vals['snr_subdominant'] = ifo_out_vals['snr_dominant']
        # IFO is stored as an int
        ifo_out_vals['ifo'] = ([event_mgr.ifo_dict[ifo]] * num_events)
        ifo_out_vals['sigma_sub_perp'] = ([1] * num_events)
        event_mgr.add_template_events_to_ifo(
            ifo, ifo_out_names, [ifo_out_vals[n] for n in ifo_out_names])
    network_out_vals['snr_2_filter'] = snr_2_filter
    network_out_vals['snr_2_filter_rss'] = network_out_vals['snr_2_filter']
    network_out_vals['nifo'] = [nifo] * num_events
    network_out_vals['timeslide_id'] = [timeslide_id] * num_events
    event_mgr.add_template_events_to_network(network_names,
        [network_out_vals[n] for n in network_names])

    event_mgr.finalize_template_events("time_index", "snr_2_filter", 
            ifo_list, args.num_timeslides)
#     finalize_template_events(event_mgr)

In [7]:
# for i, ifo in enumerate(ifo_list):
#     if_mask = event_mgr.events["ifo"] == 0
#     ifo_eid = event_mgr.events[ifo_mask]["event_id"]
#     if (np.diff(ifo_eid)).sum() != len(ifo_eid) - 1:
#         err_msg =  "Indexing as implemented won't work: ifo event ids are not ascending "
#         err_msg += "consecutive integers and so do not correspond to indices. "
#         err_msg += "To implement: Must map event_ids to indices properly."
#         raise NotImplementedError(err_msg)
#     else:
#         # Assume indexes correspond to eventids.
#         ifo_time = 

In [8]:
event_mgr.events["snr_dominant"]

array([ 6.+0.j, 18.+0.j, 12.+0.j, 18.+0.j,  6.+0.j, 18.+0.j],
      dtype=complex64)

In [9]:
event_mgr.events["event_id"]

array([0, 1, 0, 1, 0, 1])

In [20]:
ii = event_mgr.events["ifo"]==0
event_mgr.events["event_id"][ii]

array([0, 1])

In [10]:
event_mgr.network_events["snr_2_filter"]

array([12.,  8., 12.,  8.], dtype=float32)

In [11]:
event_mgr.network_events["timeslide_id"]

array([0, 0, 1, 1])

In [12]:
event_mgr.network_events["event_id"]

array([0, 1, 2, 3])

In [13]:
event_mgr.write_events("./test9.hdf")

In [14]:
event_mgr.write_performance

False

In [22]:
event_mgr.network_events

array([(0, 0, 0, 0, 0, 3, 12., 12., 0), (1, 1, 1, 0, 1, 3,  8.,  8., 0),
       (0, 0, 0, 0, 2, 3, 12., 12., 1), (1, 1, 1, 0, 3, 3,  8.,  8., 1)],
      dtype=[('H1_event_id', '<i8'), ('L1_event_id', '<i8'), ('V1_event_id', '<i8'), ('template_id', '<i8'), ('event_id', '<i8'), ('nifo', '<i8'), ('snr_2_filter', '<f4'), ('snr_2_filter_rss', '<f4'), ('timeslide_id', '<i8')])

In [24]:
event_mgr.write_to_hdf??

In [25]:
pwd

'/Users/camill/projects/pycbc/test_hm/playground'